# Day 2: Diversification opportunities and the product space

ECU Summer School of Economic Complexity. UM6P Rabat. Afternoon Session 2.

Yesterday you built every number by hand so that none of the machinery would stay a black box. Today we hand the machinery to the `ecomplexity` package and spend our time on what the numbers say. The question of the day: given what a country already makes, what is nearby?

The steps: measure how related every pair of products is (proximity), place your country on the product space, score every product by how close it sits to the current basket (density), and turn the scores into a shortlist worth arguing about. Day 1's chain still runs underneath, since everything starts from RCA.

`COUNTRY`, `COMPARATORS`, and `YEAR` at the top drive every figure. On Google Colab, save your own copy before anything else: File menu, "Save a copy in Drive".

## 0. Setup

Same data packet as Day 1, plus two small files with the Atlas product-space layout. The notebook finds the packet on its own, downloading it on first run if you are not working from the course repository.

In [ ]:
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import spearmanr

# The complexity engine for the rest of the week. We pin the development branch
# of the Growth Lab's py-ecomplexity, which accepts a custom Mcp matrix
# (presence_test="manual"), and install it on the spot if it is missing, as on
# a fresh Colab session.
try:
    from ecomplexity import ecomplexity, proximity
except ModuleNotFoundError:
    import subprocess

    subprocess.run(
        [
            "uv",
            "pip",
            "install",
            "--system",
            "git+https://github.com/harvard-growth-lab/py-ecomplexity@develop",
        ],
        check=True,
    )
    from ecomplexity import ecomplexity, proximity

# --- The ONE place you edit to make this notebook about your own country. ------
# Codes are ISO 3166-1 alpha-2 (two letters). MA = Morocco, EG = Egypt,
# TN = Tunisia, KR = Republic of Korea, DE = Germany. See reference/units.csv.
COUNTRY = "MA"  # the focus country (Morocco)
COMPARATORS = ["EG", "TN", "KR"]  # Egypt, Tunisia (MENA) + Korea (frontier)
YEAR = 2023  # latest year in the packet
# -------------------------------------------------------------------------------

LOCAL_PACKET = Path("../../data/processed/morocco_data_packet")
DATA_RELEASE = "https://github.com/shreyasgm/ecu-complexity-labs/releases/download/data-v1"

if LOCAL_PACKET.exists():
    PACKET = LOCAL_PACKET.resolve()
else:
    PACKET = Path("morocco_data_packet").resolve()
    for folder in ("exports", "reference"):
        if not (PACKET / folder).exists():
            print(f"Downloading {folder}.zip from the course data release ...")
            zip_file, _ = urllib.request.urlretrieve(f"{DATA_RELEASE}/{folder}.zip")
            with zipfile.ZipFile(zip_file) as zf:
                zf.extractall(PACKET)

# Fail loud on anything missing, naming the exact file.
required = [
    PACKET / "exports" / "outputs.parquet",
    PACKET / "exports" / "densities.parquet",
    PACKET / "reference" / "fields.csv",
    PACKET / "reference" / "units.csv",
    PACKET / "reference" / "umap_layout_hs92.csv",
    PACKET / "reference" / "top_edges_hs92.csv",
]
for f in required:
    if not f.exists():
        raise FileNotFoundError(
            f"Required file not found: {f}\n"
            "If you downloaded the packet before Day 2, delete your local "
            "morocco_data_packet folder and re-run this cell: the layout files "
            f"were added to reference.zip on 7 July. Release: {DATA_RELEASE}"
        )

exports = pd.read_parquet(PACKET / "exports" / "outputs.parquet")
fields = pd.read_csv(PACKET / "reference" / "fields.csv")
units = pd.read_csv(PACKET / "reference" / "units.csv")

# The packet stores measures as float32 to keep the download small. Upcast
# before computing, or the identity checks below fail on the last digits.
float32_cols = exports.select_dtypes("float32").columns
exports[float32_cols] = exports[float32_cols].astype("float64")

prod_name = fields.set_index("Field ID")["Field Name"]
print(f"Packet: {PACKET}")
print(f"Focus country: {COUNTRY} | comparators: {COMPARATORS} | year: {YEAR}")

## 1. Rebuild yesterday's objects

Everything today grows out of the RCA matrix from Day 1, so we rebuild it in one screen. One line is left for you to fill from memory.

In [ ]:
# Country x product matrix of export values, then shares (Day 1, section 2).
year_long = exports.query("Period == @YEAR")[["Unit", "Field ID", "Outputs"]]
X = year_long.pivot_table(index="Unit", columns="Field ID", values="Outputs", fill_value=0.0)
shares = X.div(X.sum(axis=1), axis=0)

# COMPLETION exercise: yesterday's key line. Fill in ONLY the world-share
# denominator; the RCA division after it is already written.
#
# PROMPT: `world_share` is each product's share of TOTAL world exports: the
# column sum of X divided by the grand total of X.
# TODO(core): complete the one missing line below.
rca = shares.div(world_share, axis=1)  # worked for you: share ÷ world share

# The binary presence matrix and the two counts, worked because you own them
# since yesterday. The RCA > 1 basket also defines what we highlight on the
# product space below.
mcp = (rca > 1).astype(float)
diversity = mcp.sum(axis=1)
ubiquity = mcp.sum(axis=0)

print(f"{COUNTRY} diversity:", int(diversity.loc[COUNTRY]))

In [ ]:
# Self-check: the Day 1 handshake. Same data and method must give the same
# numbers as yesterday.
assert (shares.sum(axis=1) - 1).abs().max() < 1e-9, (
    "shares must sum to 1 per country. Divide each row by its own row total."
)
_ww = X.sum(axis=1) / X.sum().sum()
assert (rca.mul(_ww, axis=0).sum(axis=0).dropna() - 1).abs().max() < 1e-6, (
    "the world-export-weighted mean of RCA must be 1 for every product (the "
    "Balassa identity). Check the world_share completion."
)
if COUNTRY == "MA" and YEAR == 2023:
    assert int(diversity.loc[COUNTRY]) == 129, (
        "Morocco's 2023 diversity was 129 yesterday and must be 129 today."
    )
print("OK: yesterday's pipeline reproduced.")

## 2. Complexity without the threshold

Day 1 ended with an exercise: move the RCA cutoff from 1 to 0.5 or to 2 and see how the presence matrix changes. Today, instead of a threshold, transform RCA into a bounded weight and run the same algebra on the weights:

$$ \widehat{M}_{cp} = \frac{\text{RCA}_{cp}}{1 + \text{RCA}_{cp}} \in [0, 1) $$

An RCA of 1 becomes 0.5 and an RCA of 9 becomes 0.9. From today on we also stop hand-rolling: the `ecomplexity` package accepts this matrix directly and returns every measure.

### Predict before running

If the threshold disappears, does the country ranking change a little or a lot? And which kind of country moves most, diversified manufacturers or concentrated resource exporters? Commit to an answer before running the next cells.

In [ ]:
# TODO(core): build the continuous presence matrix `m_hat` from `rca`.
#
# PROMPT: apply the transformation cell by cell: rca / (1 + rca). DataFrames
# divide elementwise, so this is one line with no loop.
# TODO(core): write your code here
raise NotImplementedError()

print("RCA 0.5 -> M-hat", round(0.5 / 1.5, 2), "| RCA 1 -> 0.5 | RCA 9 -> 0.9")
print(f"{COUNTRY} M-hat for Passenger Cars:", round(m_hat.loc[COUNTRY, "P - 8703"], 3))

<details>
<summary><b>Stuck? Reveal a hint (then a fuller hint).</b></summary>

Hint 1. Each cell of `m_hat` is that cell's RCA divided by one plus that cell's RCA. No sums, no axes.

Hint 2. Arithmetic on a DataFrame is elementwise, so `rca / (1 + rca)` transforms every cell at once. A cell with RCA exactly 1 must come out as 0.5.
</details>

The package wants long-format data (one row per country and product) and a dictionary naming the four columns. With `presence_test="manual"` it takes our matrix as given instead of recomputing RCA, which is how you feed it any custom presence matrix, binary or continuous.

### Fix the broken cell

The call below forgets `presence_test="manual"`, so the package treats our M-hat weights as if they were raw export values, recomputes RCA on them, and binarizes. It runs without a single error and returns a full set of plausible numbers, and they are the wrong numbers. Repair the call, then check the self-check catches the difference. This is the shape of most real package bugs: not a crash, a silent wrong answer.

In [ ]:
# Long format: one row per (country, product), value = the M-hat weight.
mhat_long = m_hat.stack().rename("val").reset_index()
mhat_long.columns = ["loc", "prod", "val"]
mhat_long["time"] = YEAR
PKG_COLS = {"time": "time", "loc": "loc", "prod": "prod", "val": "val"}

# TODO(core): the call below is missing the argument that tells the package our
# values ARE the presence matrix. Add it.
#
# PROMPT: pass presence_test="manual". Without it the package rebuilds a binary
# Mcp from the M-hat values, which is a different (and wrong) calculation.
# TODO(core): write your code here
raise NotImplementedError()

print("columns returned:", [c for c in cdata.columns if c not in mhat_long.columns])

<details>
<summary><b>Stuck? Reveal a hint (then a fuller hint).</b></summary>

Hint 1. Read the printed `mcp` column of your output. If it only contains 0s and 1s, the package binarized your weights, which is exactly the bug.

Hint 2. The docstring says: `presence_test="manual"` means M_cp is taken as given from the value column. Add that argument to the call and rerun.
</details>

In [ ]:
# Self-check: the package must have used OUR continuous weights, not a
# binarized version of them.
_mcp_col = cdata["mcp"].dropna()
assert _mcp_col.nunique() > 100, (
    "the returned mcp column is binary, so the package recomputed presence from "
    "your values. Pass presence_test='manual' so M-hat is used as given."
)
_check = cdata.set_index(["loc", "prod"])["mcp"]
_sample = m_hat.stack().sample(500, random_state=0)
assert np.allclose(_check.reindex(_sample.index).values, _sample.values), (
    "the mcp column must equal your m_hat values cell for cell."
)
print("OK: the package ran on the continuous M-hat.")

In [ ]:
# One call gave us the whole toolkit: ECI, PCI, proximity-based density, and
# the two outlook measures, all computed on the continuous matrix. Pull out the
# country-level ECI and compare it with yesterday's binary version, which we
# get from a second package call on the raw export values (the default
# presence test reproduces Day 1's hand-rolled index).
raw_long = year_long.rename(columns={"Unit": "loc", "Field ID": "prod", "Outputs": "val"}).assign(
    time=YEAR
)
cdata_bin = ecomplexity(raw_long, PKG_COLS, check_logsupermodularity=False, verbose=False)


def zscore(s: pd.Series) -> pd.Series:
    """Standardize to mean 0, sd 1, as Day 1 did for the hand-rolled index."""
    return (s - s.mean()) / s.std()


eci_con = zscore(cdata.groupby("loc")["eci"].first())
eci_bin = zscore(cdata_bin.groupby("loc")["eci"].first())
pci_con = zscore(cdata.groupby("prod")["pci"].first())

common_c = eci_bin.index.intersection(eci_con.index)
rho_bc = spearmanr(eci_bin[common_c], eci_con[common_c]).statistic
print(f"Spearman(binary ECI, continuous ECI) = {rho_bc:.3f} over {len(common_c)} countries")
print(
    f"{COUNTRY}: binary {eci_bin[COUNTRY]:.2f} (rank {int(eci_bin.rank(ascending=False)[COUNTRY])})"
    f" | continuous {eci_con[COUNTRY]:.2f} (rank {int(eci_con.rank(ascending=False)[COUNTRY])})"
)

# Signed rank shifts: negative = the country RISES once the threshold is gone.
rank_shift = eci_con[common_c].rank(ascending=False) - eci_bin[common_c].rank(ascending=False)
movers = rank_shift.reindex(rank_shift.abs().nlargest(6).index).astype(int)
print("Biggest movers (rank places; negative = rises without the threshold):")
print(movers.to_string())

In [ ]:
# Binary against continuous, one point per country.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(eci_bin[common_c], eci_con[common_c], s=12, alpha=0.4)
lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
ax.plot(lims, lims, color="grey", lw=1, ls="--", label="no change")
for code in [COUNTRY, *COMPARATORS]:
    if code in common_c:
        ax.scatter(
            eci_bin[code],
            eci_con[code],
            s=90,
            color="#ee3e4c" if code == COUNTRY else "#e5bd4f",
            zorder=5,
        )
        ax.annotate(
            code,
            (eci_bin[code], eci_con[code]),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=10,
        )
ax.set_xlabel("ECI from binary Mcp (RCA > 1)")
ax.set_ylabel("ECI from continuous M-hat (no threshold)")
ax.set_title(f"Dropping the threshold barely moves most countries ({YEAR})")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Self-check: M-hat is what the formula says, and the two rankings agree
# strongly without being identical. Perfect agreement would mean the threshold
# never mattered; weak agreement would mean the index is fragile.
assert float(m_hat.values.min()) >= 0 and float(m_hat.values.max()) < 1, (
    "M-hat must lie in [0, 1). It is rca / (1 + rca)."
)
_probe = rca.iloc[:5, :5].values
assert np.allclose(_probe / (1 + _probe), m_hat.iloc[:5, :5].values), (
    "M-hat must equal rca / (1 + rca) cell for cell."
)
assert 0.85 < rho_bc < 0.9999, (
    f"binary and continuous ECI should rank countries almost alike (got "
    f"{rho_bc:.3f}). Much lower points at the presence_test bug above."
)
if COUNTRY == "MA" and YEAR == 2023:
    assert abs(eci_bin[COUNTRY] - (-0.47)) < 0.05, (
        "the binary ECI should reproduce Day 1's Morocco value of about -0.47."
    )
print("OK: continuous complexity computed, and the rankings mostly survive.")

### Interpret (no AI)

Take the top country in your movers table and note the sign of its shift. Then think about what M-hat does to an RCA of 0.8 compared with an RCA of 0.05: the first becomes 0.44, the second almost nothing. Explain why your top mover moved in the direction it did, and what that tells you about quoting any ECI number without saying which presence rule produced it.

TODO(concept): your answer here.

## 3. Proximity, once by hand

The morning defined proximity as the smaller of two conditional probabilities: of the countries exporting one product competitively, what fraction also export the other, and vice versa. Before trusting the package with 862 products, count it once on Day 1's toy table, where you can check the answer on your fingers.

In [ ]:
# Day 1's toy table, rebuilt from the raw values.
# fmt: off
toy = pd.DataFrame(
    [[ 10,  40, 100, 300],   # Country A
     [150, 100,  10,   0],   # Country B
     [ 20, 200,   1,   0]],  # Country C
    index=["Country A", "Country B", "Country C"],
    columns=["Wheat", "Textiles", "Motorcycles", "Cell phones"],
).astype(float)
# fmt: on
toy_shares = toy.div(toy.sum(axis=1), axis=0)
toy_rca = toy_shares.div(toy.sum(axis=0) / toy.sum().sum(), axis=1)
toy_mcp = (toy_rca > 1).astype(int)
toy_ubiquity = toy_mcp.sum(axis=0)
print(toy_mcp)
print("\ntoy ubiquity:", toy_ubiquity.to_dict())

In [ ]:
# TODO(core): the proximity between Wheat and Textiles, by counting.
#
# PROMPT: count the countries whose Mcp is 1 for BOTH products (multiply the
# two columns and sum), then divide by the LARGER of the two ubiquities.
# Dividing by the larger ubiquity is the same as taking the min of the two
# conditional probabilities: same numerator, bigger denominator.
# TODO(core): write your code here
raise NotImplementedError()

print("countries exporting both:", toy_both)
print("phi(Wheat, Textiles) =", toy_phi_wt)

<details>
<summary><b>Stuck? Reveal a hint (then a fuller hint).</b></summary>

Hint 1. The product of the two Mcp columns is 1 exactly where a country exports both. Sum it to count co-exporters.

Hint 2. P(Wheat given Textiles) divides the co-export count by Textiles' ubiquity, and P(Textiles given Wheat) divides it by Wheat's. The min is the one with the larger denominator, so use `max(...)` of the two ubiquities.
</details>

In [ ]:
# Self-check: exact by construction. Only Country B exports both, Wheat's
# ubiquity is 1 and Textiles' is 2, so phi = 1 / max(1, 2) = 0.5. The package
# must agree with your fingers.
assert toy_both == 1, "exactly one toy country (Country B) exports both Wheat and Textiles."
assert abs(toy_phi_wt - 0.5) < 1e-12, (
    "phi(Wheat, Textiles) must be 0.5. If you got 1.0 you divided by the "
    "SMALLER ubiquity, which takes the max of the conditionals instead of the min."
)
toy_long = toy_mcp.astype(float).stack().rename("val").reset_index()
toy_long.columns = ["loc", "prod", "val"]
toy_long["time"] = 1
_tp = proximity(toy_long, PKG_COLS, presence_test="manual").set_index(["prod_1", "prod_2"])[
    "proximity"
]
assert abs(_tp.loc[("Wheat", "Textiles")] - 0.5) < 1e-12, (
    "the package's proximity on the toy table must equal your hand count of 0.5."
)
assert abs(_tp.loc[("Motorcycles", "Cell phones")] - 1.0) < 1e-12, (
    "phi(Motorcycles, Cell phones) must be 1.0: the one country making either makes both."
)
print("OK: hand count and package agree on the toy table.")

## 4. Proximity on the real matrix

Now the package computes the same object for all 862 products, on the continuous M-hat. The formula generalizes cleanly: weights replace the 0s and 1s, so two products are close when the same countries carry weight in both.

In [ ]:
phi_edges = proximity(mhat_long, PKG_COLS, presence_test="manual")
phi_pairs = phi_edges.set_index(["prod_1", "prod_2"])["proximity"]

pair_value = phi_pairs.loc[("P - 6106", "P - 6104")]
_upper = phi_edges.query("prod_1 < prod_2")["proximity"]
pair_pct = (_upper < pair_value).mean()
print(
    f"phi(Women's Knit Shirts, Women's Knit Suits) = {pair_value:.3f}, "
    f"higher than {pair_pct:.1%} of all pairs"
)

The lecture's flagship pair lands where it should. The value is not the 0.967 from the lecture's table, because that number came from binary Mcp on 2010 data and ours is continuous on 2023 data, but the pair still beats essentially every other pair in the matrix. The method is stable even where the numbers move.

In [ ]:
# Why the min and not the max? Ask the package for the ASYMMETRIC version and
# look at the most lopsided pair: Oxometallic Salts (yesterday's top-PCI
# product, two exporters) against Non-Aqueous Paints (42 exporters).
phi_asym = proximity(mhat_long, PKG_COLS, presence_test="manual", asymmetric=True)
pa = phi_asym.set_index(["prod_1", "prod_2"])["proximity"]
p_rare, p_common = "P - 2841", "P - 3208"
print(f"{prod_name[p_rare]} -> {prod_name[p_common]}: {pa.loc[(p_rare, p_common)]:.2f}")
print(f"{prod_name[p_common]} -> {prod_name[p_rare]}: {pa.loc[(p_common, p_rare)]:.2f}")
print(f"symmetric phi (the min): {phi_pairs.loc[(p_rare, p_common)]:.2f}")

Nearly everyone who makes Oxometallic Salts also makes paints, so one direction looks strong. Almost no paint maker produces Oxometallic Salts, so the other direction is close to zero. Taking the max would declare the pair related on the evidence of two countries; the min keeps only what holds in both directions. This is the wine and ostrich eggs problem from the lecture, found in our own data.

In [ ]:
# Self-check: structure of the proximity output.
_swapped = phi_edges.rename(columns={"prod_1": "prod_2", "prod_2": "prod_1"})
_merged = phi_edges.sample(2000, random_state=0).merge(
    _swapped, on=["prod_1", "prod_2"], suffixes=("", "_rev")
)
assert np.allclose(_merged["proximity"], _merged["proximity_rev"]), (
    "symmetric proximity must satisfy phi(a, b) == phi(b, a)."
)
assert 0 <= phi_pairs.min() and phi_pairs.max() <= 1, (
    "proximity is a probability-like weight and must lie in [0, 1]."
)
assert pair_pct > 0.99, (
    "the shirts/suits pair must land near the top of all pairs. If it is "
    "mid-ranked, check that the proximity call used presence_test='manual'."
)
print("OK: proximity is symmetric, bounded, and ranks the flagship pair on top.")

## 5. The product space

The morning showed the product space twice, and now you draw it. The node positions and edges come from the Atlas layout that ships in the packet, and the node colors below are the Atlas cluster palette from the Growth Lab design library. Your country's RCA > 1 products are highlighted; everything else fades to background.

### Predict before running

Which product family sits at the dense core of the space: oil and minerals, agriculture, or machinery and chemicals? And will your country's highlighted products sit mostly in that core or around the edge?

In [ ]:
from IPython.display import HTML  # noqa: E402  (kept next to its one use)

layout_nodes = pd.read_csv(
    PACKET / "reference" / "umap_layout_hs92.csv", dtype={"product_hs92_code": str}
)
layout_edges = pd.read_csv(PACKET / "reference" / "top_edges_hs92.csv", dtype=str)

# HS92 codes need zero-padding to 4 digits to join our Field IDs (714 -> 0714).
layout_nodes["Field ID"] = "P - " + layout_nodes["product_hs92_code"].str.zfill(4)
nodes = (
    layout_nodes.merge(fields[["Field ID", "Field Name"]], on="Field ID", how="left")
    .assign(
        world_exports=lambda d: d["Field ID"].map(X.sum(axis=0)),
        in_basket=lambda d: d["Field ID"].map(mcp.loc[COUNTRY]).fillna(0).astype(bool),
    )
    .dropna(subset=["world_exports"])
)

# Atlas product-space cluster palette (Growth Lab design library,
# csv_palettes/atlas_product_space_clusters.csv), mapped to the layout's names.
CLUSTER_COLORS = {
    "Agriculture": "#e0b614",
    "Construction, Building, and Home Supplies": "#c77c2b",
    "Electronic and Electrical Goods": "#5cc7c6",
    "Industrial Chemicals and Metals": "#9c3bd6",
    "Metalworking and Electrical Machinery and Parts": "#c43d3d",
    "Minerals": "#7a6a63",
    "Textile and Home Goods": "#8a8a8a",
    "Textile Apparel and Accessories": "#2fa84f",
}

# The edge file is mirrored (each pair appears both ways); keep each pair once.
pair_sorted = np.sort(layout_edges.values, axis=1)
pairs = pd.DataFrame(pair_sorted, columns=["a", "b"]).drop_duplicates()
xy = layout_nodes.set_index("product_hs92_code")[["product_space_x", "product_space_y"]]
edge_x, edge_y = [], []
for a, b in pairs.values:
    if a in xy.index and b in xy.index:
        edge_x += [xy.loc[a, "product_space_x"], xy.loc[b, "product_space_x"], None]
        edge_y += [xy.loc[a, "product_space_y"], xy.loc[b, "product_space_y"], None]

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=edge_x,
        y=edge_y,
        mode="lines",
        line=dict(color="#E3E3E3", width=0.6),
        hoverinfo="skip",
        showlegend=False,
    )
)
size = 3 + 15 * np.sqrt(nodes["world_exports"] / nodes["world_exports"].max())
for cluster, color in CLUSTER_COLORS.items():
    grp = nodes["product_space_cluster_name"] == cluster
    for basket, opacity, outline in [(False, 0.22, 0), (True, 1.0, 1.2)]:
        sel = grp & (nodes["in_basket"] == basket)
        if not sel.any():
            continue
        fig.add_trace(
            go.Scatter(
                x=nodes.loc[sel, "product_space_x"],
                y=nodes.loc[sel, "product_space_y"],
                mode="markers",
                name=cluster,
                legendgroup=cluster,
                showlegend=False,
                marker=dict(
                    size=size[sel],
                    color=color,
                    opacity=opacity,
                    line=dict(width=outline, color="black"),
                ),
                text=nodes.loc[sel, "Field Name"] + " (" + nodes.loc[sel, "Field ID"] + ")",
                hoverinfo="text",
            )
        )
    # A dedicated legend entry at full opacity, so the swatches stay readable
    # instead of inheriting the faded background markers.
    fig.add_trace(
        go.Scatter(
            x=[None],
            y=[None],
            mode="markers",
            name=cluster,
            legendgroup=cluster,
            showlegend=True,
            marker=dict(size=9, color=color),
        )
    )
fig.update_layout(
    title=f"The product space, {YEAR}. Highlighted: products {COUNTRY} exports competitively.",
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    plot_bgcolor="white",
    height=650,
    legend=dict(font=dict(size=10)),
)
# Embedded inline so the interactive figure survives offline export, as with
# Day 1's treemap.
HTML(fig.to_html(full_html=False, include_plotlyjs="inline"))

Read the picture before scrolling on. The dense core is machinery, chemicals, and metalworking. A highlighted product in the core has many strong neighbours; one on the periphery connects to little. Where the basket sits is a summary of what the country can reach, and the next section turns that into a number per product.

In [ ]:
# Self-check: the join worked and the highlight is the basket.
_matched = nodes["Field ID"].isin(X.columns).mean()
assert _matched > 0.95, (
    f"only {_matched:.0%} of layout products matched the trade matrix. Zero-pad "
    "the HS codes to 4 digits before prefixing with 'P - '."
)
_expected = int(mcp.loc[COUNTRY][mcp.columns.isin(nodes["Field ID"])].sum())
assert int(nodes["in_basket"].sum()) == _expected, (
    "highlighted nodes must equal the country's diversity restricted to "
    "products present in the layout."
)
print(f"OK: {_matched:.1%} of nodes joined, {int(nodes['in_basket'].sum())} highlighted.")

## 6. Density: how close is each product?

The morning's closing slide defined relatedness density as the share of a product's total relatedness that a country already holds. The package computed it for every country and product in the section 2 call, weighting each product's proximities by the country's M-hat and dividing by the product's total proximity. A density of 0.25 means a quarter of everything related to this product is already in the basket. Tomorrow this number becomes the regressor in the entry regressions, so today the job is to trust it and read it.

In [ ]:
density_ma = cdata.query("loc == @COUNTRY").set_index("prod")["density"]
in_basket = density_ma[mcp.loc[COUNTRY] == 1]
out_basket = density_ma[mcp.loc[COUNTRY] == 0]
print(
    f"{COUNTRY} density: {in_basket.mean():.3f} inside the RCA > 1 basket, "
    f"{out_basket.mean():.3f} outside it"
)

In [ ]:
# Self-check: two things density must do if it measures what it claims.
assert 0 <= density_ma.min() and density_ma.max() <= 1 + 1e-9, (
    "density is a share of total relatedness and must lie in [0, 1]."
)
assert in_basket.mean() > out_basket.mean(), (
    "products a country already exports must sit at higher average density "
    "than products it does not: relatedness concentrates around the basket. "
    "If this fails, the loc/prod columns are probably swapped."
)
print("OK: density is a bounded share and the basket sits denser than the rest.")

In [ ]:
# One outside check, as on Day 1: WIPO publishes its own relatedness density
# for the same countries and products.
wipo_density = (
    pd.read_parquet(PACKET / "exports" / "densities.parquet")
    .query("Period == @YEAR and Unit == @COUNTRY")
    .set_index("Field ID")["Relatedness Density"]
)
_common_w = density_ma.index.intersection(wipo_density.index)
rho_wipo = spearmanr(density_ma[_common_w], wipo_density[_common_w]).statistic
print(f"Spearman(our density, WIPO density) = {rho_wipo:.3f} over {len(_common_w)} products")

In [ ]:
# Self-check: agreement should be real and imperfect. WIPO uses a different
# presence rule AND a different proximity matrix, and density inherits both
# choices, so the gap here is wider than yesterday's PCI comparison.
assert 0.3 < rho_wipo < 0.95, (
    f"expected directional but imperfect agreement with WIPO (got {rho_wipo:.3f})."
    " Near 0 suggests a broken pipeline; near 1 means you loaded WIPO's column."
)
print(f"OK: real but imperfect agreement ({rho_wipo:.3f}), as expected.")

### Interpret (no AI)

Yesterday your PCI matched WIPO's at a Spearman near 0.92. Today your density matches at the weaker value printed above, and neither gap is a bug. Explain why a derived measure agrees less than the index it is built from. Name the two upstream choices density inherits, and say what that means for comparing density numbers across two reports that never state their presence rule or their proximity matrix.

TODO(concept): your answer here.

## 7. The payoff

Cross the two numbers you now have for every product the country does not yet export: density says how reachable it is, PCI says how sophisticated it is. The corner worth attention is high on both. The package's outlook measures summarize the same idea: COI adds up how much complexity sits near the whole basket, and COG says how much new reach a single product would unlock if entered.

### Predict before running

Write down one product you believe is realistically nearby for your country. Then find it, or its family, in the ranked table and note where it lands. Intuitions here usually run more high-tech than the data.

In [ ]:
ma_rows = cdata.query("loc == @COUNTRY").set_index("prod")
candidates = (
    pd.DataFrame(
        {
            "density": density_ma,
            "pci": pci_con,
            "cog": ma_rows["cog"],
            "world_exports": X.sum(axis=0),
            "rca_true": rca.loc[COUNTRY],
        }
    )
    .query("rca_true < 1")
    .dropna(subset=["pci"])
)

coi_value = float(ma_rows["coi"].iloc[0])
print(f"{COUNTRY} complexity outlook (COI): {coi_value:.2f}")

pci_median = candidates["pci"].median()
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(
    candidates["density"],
    candidates["pci"],
    s=3 + 120 * np.sqrt(candidates["world_exports"] / candidates["world_exports"].max()),
    alpha=0.35,
)
ax.axvline(candidates["density"].median(), color="grey", lw=0.8, ls=":")
ax.axhline(pci_median, color="grey", lw=0.8, ls=":")
_frontier = candidates.query("pci > @pci_median").nlargest(5, "density")
_offsets = [(8, 12), (8, -20), (8, 32), (8, -40), (8, 52)]
for (fid, row), (dx, dy) in zip(_frontier.iterrows(), _offsets):
    ax.scatter(row["density"], row["pci"], s=60, color="#ee3e4c", zorder=5)
    ax.annotate(
        str(prod_name.get(fid, fid))[:24],
        (row["density"], row["pci"]),
        xytext=(dx, dy),
        textcoords="offset points",
        fontsize=8,
        arrowprops=dict(arrowstyle="-", lw=0.5, color="grey"),
    )
ax.set_xlabel(f"Density around {COUNTRY}'s basket (share of relatedness already held)")
ax.set_ylabel("PCI (continuous, standardized)")
ax.set_title(f"Diversification frontier for {COUNTRY}, {YEAR}")
plt.tight_layout()
plt.show()

In [ ]:
# The ranked table: unexploited products above median PCI, by density. This
# and the product space are the two exhibits your team carries to Day 5.
opportunities = (
    candidates.query("pci > @pci_median")
    .nlargest(15, "density")
    .assign(product=lambda d: d.index.map(prod_name))
    .loc[:, ["product", "density", "pci", "cog", "world_exports"]]
    .assign(world_exports=lambda d: (d["world_exports"] / 1e9).round(1))
    .rename(columns={"world_exports": "world_exports_bnUSD"})
    .round({"density": 3, "pci": 2, "cog": 2})
)
opportunities

In [ ]:
# Self-check: the table is what it claims to be.
assert (candidates["rca_true"] < 1).all(), (
    "candidates must be products the country does not yet export competitively."
)
assert len(opportunities) > 0 and opportunities["density"].is_monotonic_decreasing, (
    "the table must be non-empty and sorted by density, largest first."
)
assert opportunities["cog"].notna().all(), "every row needs its COG from the package output."
print(f"OK: {len(opportunities)} ranked opportunities for {COUNTRY}.")

### Interpret (no AI)

The model ranks; you judge. Pick one product from your top ten and argue for dropping it from a shortlist you would hand to a minister, on grounds the data does not contain: e.g. water, land, geopolitics, a competitor's lock-in, an environmental cost, or something you know about your country that the trade matrix cannot. Two or three sentences. "The density is high so we keep it" is the one wrong answer.

TODO(concept): your answer here.

One caution: Proximity is revealed co-location, so it says shirt exporters tend to export suits, without saying why. Density ranks nearness; whether nearby products actually get entered more often is an empirical question. And everything here rests on trade data alone, so services, informality, and the domestic economy are missing from this analysis.

## 8. Recall: close the loop

From memory, without scrolling up:

1. Proximity takes the minimum of two conditional probabilities. Which two, and why the minimum?
2. Your country's density for some product is 0.30. Say precisely what that means.
3. What does COG tell you that density alone does not?
4. Tomorrow we regress entry on density. What sign do we expect, and what result would falsify the principle of relatedness?

TODO(concept): your answers here.

## Stretch (for fast finishers, scaffolding removed)

(a) Hand-roll what the package did. Proximity is from the binary Mcp: co-export counts `mcp.T @ mcp`, divide by ubiquity, take the elementwise min with the transpose. Density is one more line. Build them and rank-correlate against the package output.

(b) Nearest-neighbour countries. `cross_domain/unit_proximities.parquet` scores how similar two countries' capability profiles are (download `cross_domain.zip` from the data release). Find your country's ten nearest neighbours and check whether your project's comparators are the data's idea of similar.

(c) Other similarity measures. The lecture listed alternatives to the min of conditionals: Jaccard, cosine. Compute one on the Mcp columns and rank-correlate it against phi.

(d) A Day 5 teaser. `cross_domain/field_proximities.parquet` links products to technology and science fields in one network. Which technologies sit closest to your country's export strengths?

In [ ]:
# (Your stretch code here, no scaffold, no self-check.)